In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
df_day = pd.read_csv('../data/raw/day.csv')
df_hour = pd.read_csv('../data/raw/hour.csv')

#### Create a Copy of the Dataset

In [21]:
df_clean = df_hour.copy()

df_clean.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


#### Check Data Types

Before cleaning, inspect the data types.

In [22]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


dteday is stored as an object (string) instead of a datetime.

#### Convert dteday to Datetime

Pandas can now perform time-based operations such as sorting, filtering, and feature extraction.

In [23]:
df_clean["dteday"] = pd.to_datetime(df_clean["dteday"])
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday      17379 non-null  datetime64[ns]
 2   season      17379 non-null  int64         
 3   yr          17379 non-null  int64         
 4   mnth        17379 non-null  int64         
 5   hr          17379 non-null  int64         
 6   holiday     17379 non-null  int64         
 7   weekday     17379 non-null  int64         
 8   workingday  17379 non-null  int64         
 9   weathersit  17379 non-null  int64         
 10  temp        17379 non-null  float64       
 11  atemp       17379 non-null  float64       
 12  hum         17379 non-null  float64       
 13  windspeed   17379 non-null  float64       
 14  casual      17379 non-null  int64         
 15  registered  17379 non-null  int64         
 16  cnt         17379 non-

#### Create a Complete Datetime Column

The dataset stores
-Date (dteday)
-Hour (hr)
separately.<br>
We'll combine them

A single timestamp is much easier to use for:

Sorting<br>
Time-series validation<br>
Lag features<br>
Rolling windows<br>

In [24]:
df_clean["datetime"] = (
    df_clean["dteday"] +
    pd.to_timedelta(df_clean["hr"], unit="h")
)

df_clean[["dteday", "hr", "datetime"]].head()

,dteday,hr,datetime
0,2011-01-01,0,2011-01-01 00:00:00
1,2011-01-01,1,2011-01-01 01:00:00
2,2011-01-01,2,2011-01-01 02:00:00
3,2011-01-01,3,2011-01-01 03:00:00
4,2011-01-01,4,2011-01-01 04:00:00


In [25]:
#Drop dteday 
df_clean = df_clean.drop(columns=['dteday'], axis=1)
df_clean.head()

,instant,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,datetime
0,1,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00
1,2,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00
2,3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00
3,4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00
4,5,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00


#### Sort Chronologically

Time-series models require data in time order.

Sort using the datetime column.

In [26]:
df_clean = df_clean.sort_values("datetime")

Reset the index.

In [27]:
df_clean = df_clean.reset_index(drop=True)

In [28]:
df_clean[["datetime"]].head()

,datetime
0,2011-01-01 00:00:00
1,2011-01-01 01:00:00
2,2011-01-01 02:00:00
3,2011-01-01 03:00:00
4,2011-01-01 04:00:00


#### Set Datetime as Index

This makes time-series operations easier.

In [29]:
df_clean.set_index("datetime", inplace=True)
df_clean.head()

,instant,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
datetime,,,,,,,,,,,,,,,,
2011-01-01 00:00:00,1,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2011-01-01 01:00:00,2,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2011-01-01 02:00:00,3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
2011-01-01 03:00:00,4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
2011-01-01 04:00:00,5,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


#### Remove Unnecessary Columns


In [30]:
print(df_clean.columns)

Index(['instant', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='object')


##### remove instant column

instant is only a row identifier.
It carries no predictive information.

In [31]:
df_clean.drop(columns=["instant"], inplace=True)

#### Drop Leakage Columns


In [32]:
df_clean.drop(columns=["casual", "registered"], inplace=True)

The dataset defines:

cnt = casual + registered

Suppose<br>
casual = 100<br>
registered = 250<br>

Then...cnt = 350

If the model receives both casual and registered, it can simply compute:
100 + 250 = 350

It doesn't need to learn anything.

This is called data leakage because the input already contains the answer.

In [33]:
print(df_clean.columns)

Index(['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt'],
      dtype='object')


#### Save the Cleaned Dataset


In [34]:
df_clean.to_csv("../data/processed/hour_clean.csv")

### Feature Engineering

In [35]:
df = df_clean.copy()
df.columns

Index(['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt'],
      dtype='object')

#### 1.Create Cyclical Features



Think about the hours:

23 → 0<br>
These are only 1 hour apart.

But numerically,<br>
23 - 0 = 23

The model thinks they are far apart.
So we convert them into a circle using sine and cosine.

#### Hour

In [36]:
df["hr_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

#### Weekday

In [37]:
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

#### Month

In [38]:
df["month_sin"] = np.sin(2 * np.pi * df["mnth"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["mnth"] / 12)

In [39]:
df.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt,hr_sin,hr_cos,weekday_sin,weekday_cos,month_sin,month_cos
datetime,,,,,,,,,,,,,,,,,,,
2011-01-01 00:00:00,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025
2011-01-01 01:00:00,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025
2011-01-01 02:00:00,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025
2011-01-01 03:00:00,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025
2011-01-01 04:00:00,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025


### 2.Create Lag Features

#### Lag 1
Previous hour demand.

In [40]:
df["lag_1"] = df["cnt"].shift(1)

#### Lag 24

Same hour yesterday.

In [41]:
df["lag_24"] = df["cnt"].shift(24)

#### Lag 168

Same hour last week.

In [42]:
df["lag_168"] = df["cnt"].shift(168)

In [43]:
df[
[
"cnt",
"lag_1",
"lag_24",
"lag_168"
]
].head(30)

,cnt,lag_1,lag_24,lag_168
datetime,,,,
2011-01-01 00:00:00,16,NaN,NaN,NaN
2011-01-01 01:00:00,40,16.0,NaN,NaN
2011-01-01 02:00:00,32,40.0,NaN,NaN
2011-01-01 03:00:00,13,32.0,NaN,NaN
2011-01-01 04:00:00,1,13.0,NaN,NaN
2011-01-01 05:00:00,1,1.0,NaN,NaN
2011-01-01 06:00:00,2,1.0,NaN,NaN
2011-01-01 07:00:00,3,2.0,NaN,NaN
2011-01-01 08:00:00,8,3.0,NaN,NaN


### 3.Rolling Window Features

Instead of looking at one previous value,

look at the average of several previous values.

#### Rolling Mean

Previous 24 hours.

In [44]:
df["rolling_mean_24"] = (
    df["cnt"]
      .rolling(window=24)
      .mean()
)

#### Rolling Median

In [45]:
df["rolling_median_24"] = (
    df["cnt"]
      .rolling(window=24)
      .median()
)

#### Rolling Standard Deviation

In [46]:
df["rolling_std_24"] = (
    df["cnt"]
      .rolling(window=24)
      .std()
)

In [47]:
df[
[
"cnt",
"rolling_mean_24",
"rolling_std_24"
]
].tail()

,cnt,rolling_mean_24,rolling_std_24
datetime,,,
2012-12-31 19:00:00,119,110.166667,89.887208
2012-12-31 20:00:00,89,110.875000,89.640011
2012-12-31 21:00:00,90,112.666667,88.732949
2012-12-31 22:00:00,61,113.708333,87.936973
2012-12-31 23:00:00,49,113.708333,87.936973


Suppose bike demand
100,
120,
110,
130,
150
<br>
Instead of only remembering
150....the model also knows Average 122 and variability.

#### 4.Peak Hour Feature
Morning
7–9
Evening
17–19

In [48]:
df["is_peak_hour"] = (
    df["hr"].isin([7,8,9,17,18,19])
).astype(int)

In [49]:
df[
[
"hr",
"is_peak_hour"
]
].head(20)

,hr,is_peak_hour
datetime,,
2011-01-01 00:00:00,0,0
2011-01-01 01:00:00,1,0
2011-01-01 02:00:00,2,0
2011-01-01 03:00:00,3,0
2011-01-01 04:00:00,4,0
2011-01-01 05:00:00,5,0
2011-01-01 06:00:00,6,0
2011-01-01 07:00:00,7,1
2011-01-01 08:00:00,8,1


### 5.Weather Interaction




Adding Interaction Features

weather x hr

temp x hr

temp x humidity

<br>
Temperature behaves differently at different hours.
<br>
Example:<br>
30°c at 2 AM is different from 30°C at 5 PM

In [50]:
df["temp_hr"] = df["temp"] * df["hr"]

Weather interaction

In [51]:
df["weather_hr"] = (
    df["weathersit"] *
    df["hr"]
)

In [57]:
df['temp_hum'] = df['temp'] * df['hum']
df.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,...,lag_1,lag_24,lag_168,rolling_mean_24,rolling_median_24,rolling_std_24,is_peak_hour,temp_hr,weather_hr,temp_hum
datetime,,,,,,,,,,,,,,,,,,,,,
2011-01-08 07:00:00,1,0,1,7,0,6,0,2,0.16,0.1818,...,2.0,84.0,16.0,60.083333,50.5,56.721989,1,1.12,14,0.1184
2011-01-08 08:00:00,1,0,1,8,0,6,0,3,0.16,0.1818,...,9.0,210.0,40.0,51.958333,44.5,47.536237,1,1.28,24,0.1488
2011-01-08 09:00:00,1,0,1,9,0,6,0,3,0.16,0.1818,...,15.0,134.0,32.0,47.208333,37.5,44.585998,1,1.44,27,0.1488
2011-01-08 10:00:00,1,0,1,10,0,6,0,2,0.18,0.1970,...,20.0,63.0,13.0,47.125000,37.5,44.557059,0,1.80,20,0.1440
2011-01-08 11:00:00,1,0,1,11,0,6,0,2,0.20,0.1818,...,61.0,67.0,1.0,46.916667,37.5,44.471698,0,2.20,22,0.1380


These interactions allow the model to learn more complex relationships between weather and time.

#### 6.Remove Missing Values

Lag and rolling features introduce missing values.

In [53]:
df.isnull().sum()

season                 0
yr                     0
mnth                   0
hr                     0
holiday                0
weekday                0
workingday             0
weathersit             0
temp                   0
atemp                  0
hum                    0
windspeed              0
cnt                    0
hr_sin                 0
hr_cos                 0
weekday_sin            0
weekday_cos            0
month_sin              0
month_cos              0
lag_1                  1
lag_24                24
lag_168              168
rolling_mean_24       23
rolling_median_24     23
rolling_std_24        23
is_peak_hour           0
temp_hr                0
weather_hr             0
temp_hum               0
dtype: int64

In [54]:
df = df.dropna()

#### 7. Verify Final Dataset

In [55]:
df.shape
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17211 entries, 2011-01-08 07:00:00 to 2012-12-31 23:00:00
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   season             17211 non-null  int64  
 1   yr                 17211 non-null  int64  
 2   mnth               17211 non-null  int64  
 3   hr                 17211 non-null  int64  
 4   holiday            17211 non-null  int64  
 5   weekday            17211 non-null  int64  
 6   workingday         17211 non-null  int64  
 7   weathersit         17211 non-null  int64  
 8   temp               17211 non-null  float64
 9   atemp              17211 non-null  float64
 10  hum                17211 non-null  float64
 11  windspeed          17211 non-null  float64
 12  cnt                17211 non-null  int64  
 13  hr_sin             17211 non-null  float64
 14  hr_cos             17211 non-null  float64
 15  weekday_sin        17211 non-null  

#### 8.Save Feature-Engineered Dataset


In [58]:
df.to_csv(
    "../data/processed/hour_featured.csv"
)